In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
views      = pd.read_csv("/home/zizo/MyHouse/CMP Year 4/Semester 2/Data Science/Project/dataset/Diginetica/train-item-views.csv", sep=";")
purchases  = pd.read_csv("/home/zizo/MyHouse/CMP Year 4/Semester 2/Data Science/Project/dataset/Diginetica/train-purchases.csv", sep=";")
products   = pd.read_csv("/home/zizo/MyHouse/CMP Year 4/Semester 2/Data Science/Project/dataset/Diginetica/products.csv", sep=";")
categories = pd.read_csv("/home/zizo/MyHouse/CMP Year 4/Semester 2/Data Science/Project/dataset/Diginetica/product-categories.csv", sep=";")
clicks    = pd.read_csv("/home/zizo/MyHouse/CMP Year 4/Semester 2/Data Science/Project/dataset/Diginetica/train-clicks.csv", sep=";")

In [3]:
queries = pd.read_csv("/home/zizo/MyHouse/CMP Year 4/Semester 2/Data Science/Project/dataset/Diginetica/train-queries.csv", sep=";")

/tmp/ipykernel_5899/1557416354.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  queries = pd.read_csv("/home/zizo/MyHouse/CMP Year 4/Semester 2/Data Science/Project/dataset/Diginetica/train-queries.csv", sep=";")


## Purchases

In [4]:
purchases.head()

,sessionId,userId,timeframe,eventdate,ordernumber,itemId
0,150,18278.0,17100868,2016-05-06,16421,25911
1,151,NaN,6454547,2016-05-06,16290,175874
2,156,7.0,1721689387,2016-05-27,21173,35324
3,179,NaN,343001,2016-05-09,16924,31233
4,246,34.0,2311046,2016-05-09,16936,34677


In [5]:
purchases.isnull().sum() / len(purchases) * 100

sessionId       0.00000
userId         62.81276
timeframe       0.00000
eventdate       0.00000
ordernumber     0.00000
itemId          0.00000
dtype: float64

In [6]:
purchases.groupby("sessionId").agg({
    "ordernumber": ["count"]
}).reset_index().sort_values(
    by=("ordernumber", "count"),
    ascending=False
).head()

,sessionId,ordernumber
,,count
357,13366,59
2157,64786,22
579,20600,22
7708,272334,20
6719,225946,19


## Views

In [7]:
views.head()

,sessionId,userId,itemId,timeframe,eventdate
0,1,NaN,81766,526309,2016-05-09
1,1,NaN,31331,1031018,2016-05-09
2,1,NaN,32118,243569,2016-05-09
3,1,NaN,9654,75848,2016-05-09
4,1,NaN,32627,1112408,2016-05-09


In [8]:
session_items = views.groupby("sessionId")["itemId"].apply(list).reset_index()

In [9]:
session_items.head()

,sessionId,itemId
0,1,"[81766, 31331, 32118, 9654, 32627, 33043, 1235..."
1,2,"[3147, 32754, 100747, 32971, 10657, 35606, 356..."
2,4,[13931]
3,5,"[5140, 35472]"
4,6,"[73191, 382683]"


In [10]:
views.isnull().sum() / len(views) * 100

sessionId     0.00000
userId       69.80759
itemId        0.00000
timeframe     0.00000
eventdate     0.00000
dtype: float64

In [11]:
views.groupby(["sessionId", "itemId"]).agg({
    "timeframe": ["count"]
}).reset_index().sort_values(
    by=("timeframe", "count"),
    ascending=False
).head()

,sessionId,itemId,timeframe
,,,count
261515,100956,4099,22
727032,316456,31999,20
480550,189739,9476,17
235185,91362,34611,16
142777,58930,9186,15


In [12]:
# clicks_views = clicks.merge(views, on=["sessionId", "itemId"], how="left")

## Products

In [13]:
products.head()

,itemId,pricelog2,product.name.tokens
0,1,10,"4875,776,56689,18212,18212,4896"
1,69585,6,"7583,18117,41805,41805,2371"
2,90939,6,"604,18117,41805,41805,2371"
3,69586,0,"2936,18117,41805,41805,2371"
4,30029,7,"4668,41805,41805,56652"


## Categories

In [14]:
categories.head()

,itemId,categoryId
0,139578,1096
1,417975,1096
2,291805,1096
3,396921,1096
4,159257,1096


In [15]:
items_categories = products.merge(categories, on="itemId", how="inner")

In [16]:
assert len(items_categories) == len(products), "They are not the same"
assert len(items_categories) == len(categories), "They are not the same"

In [17]:
items_categories.head()

,itemId,pricelog2,product.name.tokens,categoryId
0,1,10,"4875,776,56689,18212,18212,4896",684
1,69585,6,"7583,18117,41805,41805,2371",176
2,90939,6,"604,18117,41805,41805,2371",176
3,69586,0,"2936,18117,41805,41805,2371",176
4,30029,7,"4668,41805,41805,56652",176


## Queries

In [18]:
queries.head()

,queryId,sessionId,userId,timeframe,duration,eventdate,searchstring.tokens,categoryId,items,is.test
0,1,1,NaN,16327074,311,2016-05-09,"16655,244087,51531,529597,58153",0,"7518,71,30311,7837,30792,8252,81766,9338,62220...",False
1,2,2,NaN,705527,314,2016-05-09,"528941,529116",0,"70095,15964,8627,134850,32754,100747,74771,314...",False
2,3,3,NaN,0,502,2016-05-09,"133713,16655,138389",0,"59081,51125,9338,9550,32087,62793,2717,10403,3...",True
3,4,4,NaN,0,1092,2016-05-09,"3918,3822,460416,528812,5276,529517,528738",0,"46632,57465,79064,57748,6080,35997,47088,6078,...",False
4,5,5,NaN,102700,266,2016-05-09,"529223,199482",0,"27312,84626,12621,46209,5140,57539,5368,12923,...",False


In [19]:
# queries[queries['sessionId'] == 1]

In [20]:
queries.groupby("sessionId").agg({
    "queryId": ['count']
}).reset_index().sort_values(
    by=("queryId", "count"),
    ascending=False
).head()

,sessionId,queryId
,,count
48161,48445,341
51278,51674,163
20278,20281,145
2871,2873,137
25669,25672,95


In [21]:
queries['queryId'].nunique() / len(queries) * 100

100.0

## Clicks

In [22]:
clicks.head()

,queryId,timeframe,itemId
0,1,16338861,24857
1,46255,16404912,30792
2,46689,3831948,8252
3,46731,16273568,33969
4,46748,4058493,7837


In [23]:
clicks[clicks['queryId'] == 1]

,queryId,timeframe,itemId
0,1,16338861,24857


In [24]:
temp = clicks.groupby(["queryId", "itemId"]) \
    .size() \
    .reset_index(name="count") \
    .sort_values(by="count", ascending=False) \

# temp[temp['queryId'] == 856206]

In [25]:
temp[temp['queryId'] == 698893]

,queryId,itemId,count
745401,698893,14519,2
745535,698893,375756,1
745536,698893,382396,1
745537,698893,382548,1
745538,698893,385601,1
...,...,...,...
745415,698893,19145,1
745416,698893,19231,1
745417,698893,24298,1
745418,698893,24307,1


In [26]:
temp['queryId'].value_counts()

queryId
698893    200
884524    197
776747    186
60819     159
275053    132
         ... 
794233      1
794231      1
794176      1
794227      1
794202      1
Name: count, Length: 633732, dtype: int64

## Notes
- queryId float
- searchstring.tokens
- product.name.tokens

In [27]:
queries_clicks = queries.merge(clicks, on="queryId", how="left")

In [28]:
queries_clicks.head()

,queryId,sessionId,userId,timeframe_x,duration,eventdate,searchstring.tokens,categoryId,items,is.test,timeframe_y,itemId
0,1,1,NaN,16327074,311,2016-05-09,"16655,244087,51531,529597,58153",0,"7518,71,30311,7837,30792,8252,81766,9338,62220...",False,16338861.0,24857.0
1,2,2,NaN,705527,314,2016-05-09,"528941,529116",0,"70095,15964,8627,134850,32754,100747,74771,314...",False,721764.0,36246.0
2,3,3,NaN,0,502,2016-05-09,"133713,16655,138389",0,"59081,51125,9338,9550,32087,62793,2717,10403,3...",True,NaN,NaN
3,4,4,NaN,0,1092,2016-05-09,"3918,3822,460416,528812,5276,529517,528738",0,"46632,57465,79064,57748,6080,35997,47088,6078,...",False,20684.0,13931.0
4,5,5,NaN,102700,266,2016-05-09,"529223,199482",0,"27312,84626,12621,46209,5140,57539,5368,12923,...",False,118873.0,35472.0


In [29]:
len(queries_clicks) / len(queries)

1.5351723002360456

In [30]:
s = queries[queries['queryId'] == 1]['items'].values[0]
nums = list(map(int, s.split(',')))
nums

[7518,
 71,
 30311,
 7837,
 30792,
 8252,
 81766,
 9338,
 62220,
 52393,
 2712,
 32902,
 33969,
 66828,
 4615,
 4751,
 12352,
 78214,
 24857,
 13682]

In [31]:
mini_query_clicks = queries_clicks.sample(1000, random_state=42).copy()

In [32]:
mini_query_clicks["items_list"] = mini_query_clicks["items"].apply(
    lambda x: list(map(int, x.split(","))) if pd.notnull(x) else []
)

mini_query_clicks["exists"] = mini_query_clicks.apply(
    lambda row: row["itemId"] in row["items_list"],
    axis=1
)

## Combining all

In [33]:
views_purchases = views.merge( 
    purchases, 
    on=["sessionId", "itemId"], how="left",
    suffixes=("_view", "_purchase"))

In [34]:
views_purchases.head()

,sessionId,userId_view,itemId,timeframe_view,eventdate_view,userId_purchase,timeframe_purchase,eventdate_purchase,ordernumber
0,1,NaN,81766,526309,2016-05-09,NaN,NaN,NaN,NaN
1,1,NaN,31331,1031018,2016-05-09,NaN,NaN,NaN,NaN
2,1,NaN,32118,243569,2016-05-09,NaN,NaN,NaN,NaN
3,1,NaN,9654,75848,2016-05-09,NaN,NaN,NaN,NaN
4,1,NaN,32627,1112408,2016-05-09,NaN,NaN,NaN,NaN


In [35]:
items_purchased = views_purchases.merge(products, 
                        on="itemId", how="left",
                        suffixes=("_view", "_products")).\
    merge(categories, on="itemId", how="left", suffixes=("_products", "_categories"))

In [36]:
items_purchased.head()

,sessionId,userId_view,itemId,timeframe_view,eventdate_view,userId_purchase,timeframe_purchase,eventdate_purchase,ordernumber,pricelog2,product.name.tokens,categoryId
0,1,NaN,81766,526309,2016-05-09,NaN,NaN,NaN,NaN,7,"119543,27945,27945",1010
1,1,NaN,31331,1031018,2016-05-09,NaN,NaN,NaN,NaN,7,"168703,22138,22138",1010
2,1,NaN,32118,243569,2016-05-09,NaN,NaN,NaN,NaN,8,"244318,27945,27945",1010
3,1,NaN,9654,75848,2016-05-09,NaN,NaN,NaN,NaN,8,"245379,27945,27945",1010
4,1,NaN,32627,1112408,2016-05-09,NaN,NaN,NaN,NaN,7,"776,295900,93396,93396",1010


In [37]:
search_activity = clicks.merge(queries, on="queryId", how="left", suffixes=("_click", "_query"))

In [38]:
search_activity.head()

,queryId,timeframe_click,itemId,sessionId,userId,timeframe_query,duration,eventdate,searchstring.tokens,categoryId,items,is.test
0,1,16338861,24857,1,NaN,16327074,311,2016-05-09,"16655,244087,51531,529597,58153",0,"7518,71,30311,7837,30792,8252,81766,9338,62220...",False
1,46255,16404912,30792,1,NaN,16396680,999,2016-05-09,"16655,244087,51531,529597,58153",0,"7518,71,30311,7837,30792,8252,81766,9338,62220...",False
2,46689,3831948,8252,1,NaN,3792515,492,2016-05-09,"244087,535273",0,"30254,7837,513,70894,8252,26612,8867,8882,9300...",False
3,46731,16273568,33969,1,NaN,16208824,597,2016-05-09,"16655,244087,51531,529597,58153",0,"7518,71,30311,7837,30792,8252,81766,9338,62220...",False
4,46748,4058493,7837,1,NaN,4047551,681,2016-05-09,"244087,535273",0,"30254,7837,513,70894,8252,26612,8867,8882,9300...",False


In [39]:
final_df = search_activity.merge(items_purchased, on=["sessionId", "itemId"], how="left", suffixes=("_search", "_items_purchased"))

In [40]:
# final_df.to_csv("final_df.csv", index=False)

In [41]:
final_df.head()

,queryId,timeframe_click,itemId,sessionId,userId,timeframe_query,duration,eventdate,searchstring.tokens,categoryId_search,...,userId_view,timeframe_view,eventdate_view,userId_purchase,timeframe_purchase,eventdate_purchase,ordernumber,pricelog2,product.name.tokens,categoryId_items_purchased
0,1,16338861,24857,1,NaN,16327074,311,2016-05-09,"16655,244087,51531,529597,58153",0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,46255,16404912,30792,1,NaN,16396680,999,2016-05-09,"16655,244087,51531,529597,58153",0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,46689,3831948,8252,1,NaN,3792515,492,2016-05-09,"244087,535273",0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,46731,16273568,33969,1,NaN,16208824,597,2016-05-09,"16655,244087,51531,529597,58153",0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,46748,4058493,7837,1,NaN,4047551,681,2016-05-09,"244087,535273",0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
